# CeyMo variants on Colab GPU (one variant + seed per session)

Runs **one variant per session**. Same recipe, seeds, batch and data split as `run_all.sh`.

`FLIP` and `IMGSZ` in cell 2 must match the GPU machine's `run_all.sh` (defaults: `FLIP=swap`, `IMGSZ=640`). Runs and results go to `runs_<tag>` / `results_<tag>` on Drive, so older runs are never mixed in.

**Once, before the first session:** in Google Drive make the folder `MyDrive/msem_pr/` and upload:
1. `experiments_for_colab.zip` (the code; it sits next to the `experiments` folder)
2. the **official CeyMo** files `train.zip` and `test.zip` (as downloaded, do not unzip)

Runtime -> Change runtime type -> **GPU**. Set `VARIANT` in cell 2, then Run all.
Training writes straight to Drive: if Colab disconnects, just Run all again and it resumes from `last.pt`.


In [ ]:
# 1) GPU + Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Settings: change VARIANT / SEED each session
VARIANT = "full"
SEED    = 0
FLIP    = "swap"  # swap = mirror + swap LA/RA, SLA/SRA labels (fix); "off"; "plain" = original
IMGSZ   = 640     # 1280 for thin markings (then BATCH = 8 if GPU memory runs out)
BATCH   = 16      # must match BATCH of the remote run_all.sh
WORKERS = 2       # Colab has 2 CPU cores
FPS     = False   # paper FPS should come from one GPU for all variants; leave False here

DRIVE      = "/content/drive/MyDrive/msem_pr"
EXP_ZIP    = f"{DRIVE}/experiments_for_colab.zip"
CEYMO_ZIPS = [f"{DRIVE}/train.zip", f"{DRIVE}/test.zip"]  # official CeyMo release, as downloaded
DATA_CACHE = f"{DRIVE}/ceymo_official_converted.zip"   # made automatically in the first session
TAG        = f"flip{FLIP}_{IMGSZ}"  # same folder names as run_all.sh
RUNS       = f"{DRIVE}/runs_{TAG}"      # copy to experiments/runs_<tag> on the remote machine
RESULTS    = f"{DRIVE}/results_{TAG}"   # copy to experiments/results_<tag> on the remote machine
EXP        = "/content/experiments"
DATA       = f"{EXP}/datasets/ceymo_official"
assert FLIP in ("swap", "off", "plain")
assert VARIANT in ("baseline", "baseline_p256", "msem_only", "pr_only", "full")
import os; os.makedirs(RUNS, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

In [ ]:
# 3) Code + Ultralytics (same version as the paper)
!pip -q install ultralytics==8.4.90 pyyaml opencv-python matplotlib
!rm -rf {EXP} && unzip -q -o "{EXP_ZIP}" -d /content
%cd {EXP}
!python patch_ultralytics.py
!python make_variant_yamls.py

In [ ]:
# 4) Dataset: the first session converts train.zip + test.zip and caches the result on Drive;
#    later sessions only unzip the cache. The split is seeded, so it matches the remote machine.
import glob, json, subprocess
if not os.path.exists(f"{DATA}/data.yaml"):
    if os.path.exists(DATA_CACHE):
        subprocess.run(["unzip", "-q", "-o", DATA_CACHE, "-d", f"{EXP}/datasets"], check=True)
    else:
        subprocess.run("rm -rf /content/ceymo_raw && mkdir -p /content/ceymo_raw", shell=True, check=True)
        for z in CEYMO_ZIPS:
            subprocess.run(["unzip", "-q", z, "-d", "/content/ceymo_raw"], check=True)
        roots = [os.path.dirname(p) for p in glob.glob("/content/ceymo_raw/**/train", recursive=True)
                 if os.path.isdir(os.path.join(os.path.dirname(p), "test"))]
        assert roots, "train/ and test/ not found after unzipping train.zip and test.zip"
        print("CeyMo root:", roots[0])
        subprocess.run(["python", "convert_ceymo.py", "--ceymo", roots[0], "--out", DATA], check=True)
        subprocess.run(["zip", "-q", "-r", DATA_CACHE, "ceymo_official"], cwd=f"{EXP}/datasets", check=True)
print(open(f"{DATA}/data.yaml").read())
s = json.load(open(f"{DATA}/split_stats.json"))
print({k: s[k]["images"] for k in ("train", "val", "test")}, "<- expected 1770 / 316 / 788")

In [ ]:
# 5) Train (resumes from last.pt if an earlier session was cut off; skips if DONE exists)
!python train_variant.py --variant {VARIANT} --data {DATA}/data.yaml --seed {SEED} \
    --batch {BATCH} --device 0 --workers {WORKERS} --project "{RUNS}" \
    --imgsz {IMGSZ} --flip {FLIP}
!ls "{RUNS}/{VARIANT}_s{SEED}" "{RUNS}/{VARIANT}_s{SEED}/weights" 

In [ ]:
# 6) Evaluate on val and test (same protocol as run_all.sh)
W = f"{RUNS}/{VARIANT}_s{SEED}/weights/best.pt"
for split in ("val", "test"):
    fps = "--fps" if (FPS and split == "test") else ""
    !python evaluate.py --weights "{W}" --data {DATA}/data.yaml --split {split} --imgsz {IMGSZ} \
        --variant {VARIANT}_s{SEED} --out "{RESULTS}" --device 0 --workers {WORKERS} {fps}
!python per_class_report.py "{RESULTS}"

In [ ]:
# 7) Optional: one zip with this variant's run + results, for downloading to the remote machine
!cd "{DRIVE}" && zip -q -r "{VARIANT}_s{SEED}_{TAG}_colab.zip" "runs_{TAG}/{VARIANT}_s{SEED}" results_{TAG}/{VARIANT}_s{SEED}_*.json results_{TAG}/ultra/{VARIANT}_s{SEED}_*
print("Saved:", f"{DRIVE}/{VARIANT}_s{SEED}_{TAG}_colab.zip")